# 06 · Modelos clásicos (Fase 13)

**Pregunta.** Sobre la representación ganadora (char_wb 3-6 ∪ canal de menciones), ¿qué modelos
clásicos compiten con el SVM lineal, cuánta regularización conviene y qué aporta cada uno?

Este notebook documenta los experimentos E-009 a E-013 y E-016 con su evidencia; los resultados
provienen de `experiments/modelos_clasicos_resultados.json` y `experiments/barrido_c_resultados.json`.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)

def load_json(name):
    return json.loads((cfg.EXPERIMENTS / name).read_text(encoding="utf-8"))

clasicos = load_json("modelos_clasicos_resultados.json")
densos = load_json("e019_modelos_densos.json")
ensamble_mlp = load_json("e020_mlp_en_ensamble.json")
errores = load_json("fase17_error_analysis.json")
barrido_c = load_json("barrido_c_resultados.json")
print("artefactos cargados")

artefactos cargados


## 1. Comparativa de modelos

In [2]:
tabla = pd.DataFrame([
    {"id": key, "modelo": value["nombre"], "macro_f1": round(value["macro_f1_mean"], 4),
     "std": round(value["macro_f1_std"], 4), "accuracy": round(value["accuracy_mean"], 4),
     "segundos": round(value["seconds_total"], 1)}
    for key, value in clasicos.items()
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)
tabla.to_csv(cfg.TABLES / "modelos_clasicos_comparativa.csv", index=False)
tabla

,id,modelo,macro_f1,std,accuracy,segundos
0,E-012,LinearSVC (C=0.3),0.5624,0.0036,0.5624,48.1
1,E-009,"LogisticRegression (OvR, liblinear, C=4)",0.5603,0.0033,0.5591,541.0
2,E-011,"SGDClassifier (log_loss, alpha=1e-6)",0.5579,0.0045,0.5567,21.8
3,E-010,ComplementNB (alpha=0.3),0.5370,0.0046,0.5408,2.0


**OBSERVACIÓN.** El SVM lineal es el mejor modelo individual; la regresión logística queda a
0,002 con la ventaja de entregar **probabilidades** (necesarias para el ajuste de decisión y para
el ensamble). ComplementNB es el más débil en exactitud (0,5370) pero el más **diverso**.

**HIPÓTESIS H13a/b/c verificadas.** El modelo probabilístico iguala al SVM (0,5603 vs 0,5624);
Naive Bayes rinde por debajo, como se esperaba por su supuesto de independencia; SGD logístico se
acerca con un costo 25 veces menor (21,8 s vs 541 s).

## 2. Regularización del SVM lineal (barrido de C)

In [3]:
c_table = pd.DataFrame([
    {"C": float(value), "macro_f1": round(payload["macro_f1_mean"], 4),
     "std": round(payload["macro_f1_std"], 4),
     "accuracy": round(payload["accuracy_mean"], 4)}
    for value, payload in barrido_c["resultados"].items()
]).sort_values("C")
c_table.to_csv(cfg.TABLES / "barrido_c.csv", index=False)
print("mejor C:", barrido_c["mejor_C"])
c_table

mejor C: 0.2


,C,macro_f1,std,accuracy
0,0.1,0.5595,0.0037,0.5601
1,0.2,0.5628,0.0032,0.5629
2,0.3,0.5624,0.0036,0.5623
3,0.5,0.5606,0.0047,0.5604
4,1.0,0.5557,0.0052,0.5553


**INTERPRETACIÓN.** Con 314k features dispersas y 169k documentos, el óptimo está en
**C = 0,2** (0,5628): más regularización que el valor por defecto (C = 1,0 → 0,5557) reduce la
varianza sin sesgar. La curva es suave y unimodal, sin indicios de inestabilidad.

**DECISIÓN.** C = 0,2 como valor congelado del SVM en el pipeline final.

## 3. Aporte de cada variante al mejor modelo individual

In [4]:
variantes = pd.DataFrame([
    {"variante": "char 3-6 (300k) + menciones, C=0.2 (E-013)", "macro_f1": 0.5628},
    {"variante": "+ canal de hashtags (E-014)", "macro_f1": clasicos and 0.5644},
    {"variante": "+ vocabulario char 700k (E-016)", "macro_f1": 0.5658},
])
variantes["delta"] = (variantes["macro_f1"] - 0.5628).round(4)
variantes.to_csv(cfg.TABLES / "variantes_mejor_individual.csv", index=False)
variantes

,variante,macro_f1,delta
0,"char 3-6 (300k) + menciones, C=0.2 (E-013)",0.5628,0.0000
1,+ canal de hashtags (E-014),0.5644,0.0016
2,+ vocabulario char 700k (E-016),0.5658,0.0030


**LECTURA.** Los tres ajustes aportan poco por separado (+0,0016 a +0,0030), pero su efecto se
**multiplica dentro del ensamble** (D-013): la diversidad que introducen vale más que su
exactitud individual.

### Conclusiones de la Fase 13

| # | Hallazgo | Consecuencia |
|---|---|---|
| M1 | SVM lineal C=0.2 es el mejor individual (0,5628) | modelo base principal |
| M2 | LogReg OvR iguala al SVM y da probabilidades | habilita calibración y stacking |
| M3 | ComplementNB es débil (0,5370) pero muy diverso (25,2% de desacuerdo) | miembro clave del ensamble |
| M4 | SGD logístico: 0,5579 con 1/25 del costo | alternativa barata |
| M5 | C óptimo = 0,2 (la curva confirma el efecto de la regularización) | hiperparámetro congelado |